<a href="https://colab.research.google.com/github/georgmandryka-cyber/lager-chatbot-rag/blob/main/lager_chatbot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q anthropic

import anthropic
from google.colab import userdata

# 1. Wissen: Arbeitsanweisungen als Textstücke (Chunks)
chunks = [
    {"thema": "Gefahrgut", "quelle": "gefahrgut.txt",
     "text": "Gefahrgut wird nur im gekennzeichneten Bereich C gelagert."},
    {"thema": "Gefahrgut", "quelle": "gefahrgut.txt",
     "text": "Entzündbare Stoffe dürfen nicht über 2 Paletten hoch gestapelt werden."},
    {"thema": "Gefahrgut", "quelle": "gefahrgut.txt",
     "text": "Bei einem Leck wird sofort der Bereich abgesperrt und der Schichtleiter informiert."},
    {"thema": "Stapler", "quelle": "stapler.txt",
     "text": "Vor jeder Schicht prüft der Fahrer Gabeln, Bremsen und Hupe."},
    {"thema": "Stapler", "quelle": "stapler.txt",
     "text": "Mängel werden sofort dem Schichtleiter gemeldet, der Stapler wird gesperrt."},
    {"thema": "Wareneingang", "quelle": "wareneingang.txt",
     "text": "Bei Abweichungen von mehr als 2 Prozent wird der Schichtleiter informiert."},
    {"thema": "Verkehr", "quelle": "verkehr.txt",
     "text": "Die Höchstgeschwindigkeit in der Halle beträgt 10 km/h."},
]

# 2. Retrieval: Stichwortsuche mit Wortanfängen
stopwoerter = {"jeder", "jede", "muss", "wird", "werden", "wann", "welche", "welcher"}
SCHWELLE = 2

def suche(frage, n=3):
    woerter = [w.strip("?.,!").lower() for w in frage.split()]
    staemme = [w[:4] for w in woerter if len(w) >= 5 and w not in stopwoerter]
    treffer = []
    for c in chunks:
        text = (c["thema"] + " " + c["text"]).lower()
        punkte = sum(1 for s in staemme if s in text)
        if punkte > 0:
            treffer.append((punkte, c))
    treffer.sort(key=lambda t: t[0], reverse=True)
    return treffer[:n]

# 3. Generation: Claude antwortet nur aus den Treffern
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

SYSTEM = (
    "Du bist ein Assistent für Lager-Arbeitsanweisungen. "
    "Antworte kurz und auf Deutsch, nur anhand der gegebenen Auszüge. "
    "Steht die Antwort nicht in den Auszügen, sag das ehrlich. "
    "Nenne am Ende die Quelle."
)

def antworten(frage):
    treffer = suche(frage)
    if not treffer or treffer[0][0] < SCHWELLE:
        return "Dazu habe ich nichts in den Unterlagen gefunden."
    kontext = "\n".join(f"- ({c['quelle']}) {c['text']}" for _, c in treffer)
    antwort = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        system=SYSTEM,
        messages=[{"role": "user",
                   "content": f"Auszüge:\n{kontext}\n\nFrage: {frage}"}],
    )
    return antwort.content[0].text

# 4. Test
fragen = [
    "Was muss ich vor jeder Schicht am Stapler prüfen?",
    "Wo wird Gefahrgut gelagert?",
    "Wie hoch darf ich entzündbare Stoffe stapeln?",
    "Wie hoch ist die Höchstgeschwindigkeit in der Halle?",
    "Wie schnell darf man in der Halle fahren?",   # Schwäche der Stichwortsuche
    "Wer ist der Bundeskanzler?",                  # steht nicht in den Unterlagen
]

for frage in fragen:
    print("FRAGE:", frage)
    print(antworten(frage))
    print()

FRAGE: Was muss ich vor jeder Schicht am Stapler prüfen?
## Prüfung vor jeder Schicht

Vor jeder Schicht musst du folgende Punkte am Stapler prüfen:

- **Gabeln**
- **Bremsen**
- **Hupe**

Werden Mängel festgestellt, müssen diese **sofort dem Schichtleiter gemeldet** werden – der Stapler wird dann gesperrt.

*Quelle: stapler.txt*

FRAGE: Wo wird Gefahrgut gelagert?
Gefahrgut wird im **gekennzeichneten Bereich C** gelagert.

*Quelle: gefahrgut.txt*

FRAGE: Wie hoch darf ich entzündbare Stoffe stapeln?
**Entzündbare Stoffe dürfen maximal 2 Paletten hoch gestapelt werden.**

Quelle: gefahrgut.txt

FRAGE: Wie hoch ist die Höchstgeschwindigkeit in der Halle?
Die Höchstgeschwindigkeit in der Halle beträgt **10 km/h**.

*Quelle: verkehr.txt*

FRAGE: Wie schnell darf man in der Halle fahren?
Dazu habe ich nichts in den Unterlagen gefunden.

FRAGE: Wer ist der Bundeskanzler?
Dazu habe ich nichts in den Unterlagen gefunden.

